<a href="https://colab.research.google.com/github/sonukumari-3355/NLP_practice_EmotionSet/blob/main/emotion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from warnings import filterwarnings
filterwarnings('ignore')

In [95]:
data=pd.read_csv("/train.txt",sep=";",header=None,names=["text","emotion"])

In [96]:
data.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [97]:
data.isnull().sum()

,0
text,0
emotion,0


In [98]:
unique_emotion=data['emotion'].unique()


In [99]:
emotion_numbers={}
i=0
for em in unique_emotion:
  emotion_numbers[em]=i
  i+=1
data['emotion']=data['emotion'].map(emotion_numbers)

In [100]:
data

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [101]:
data["text"]=data["text"].apply(lambda x:x.lower())

In [102]:
import string
def remove_punc(txt):
  return txt.translate(str.maketrans("","",string.punctuation))

In [103]:
data["text"]=data["text"].apply(remove_punc)

In [104]:
def remove_numbers(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new+=i
  return new
data["text"]=data["text"].apply(remove_numbers)

In [105]:
def remove_emoji(txt):
  new=""
  for i in txt:
    if i.isascii():
      new+=i
  return new
data["text"]=data["text"].apply(remove_emoji)

In [106]:
def remove_url(txt):
  new=""
  for i in txt.split():
    if not i.startswith("http"):
      new+=i+" "
  return new
data["text"]=data["text"].apply(remove_url)

In [107]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [108]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [109]:
stop_words=set(stopwords.words("english"))


In [110]:
data.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake '

In [111]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)

In [112]:
data['text'] = data['text'].apply(remove)

In [113]:
data.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [114]:
data.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [115]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data['text'], data['emotion'], test_size=0.20, random_state=42)


In [116]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))


0.76875


In [117]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [118]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)


nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)


MultinomialNB()

In [119]:
y_pred = nb2_model.predict(X_test_tfidf)

In [120]:
print(accuracy_score(y_test, y_pred))

0.66125


In [121]:
from sklearn.linear_model import LogisticRegression

In [122]:
logistic_model = LogisticRegression(max_iter=1000)

In [123]:
logistic_model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [124]:
log_pred = logistic_model.predict(X_test_tfidf)

In [125]:
print(accuracy_score(y_test,log_pred ))

0.8621875
